<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/07_reward_oracle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective :Reward oracle for bandits simulator





Given: Features ↓ Incremental delivery duration

In [5]:
local = False

# Configure SIMD before importing NumPy/SciPy when running locally on AVX-512 hardware.
if local:
    import os
    os.environ.setdefault('MKL_ENABLE_INSTRUCTIONS', 'AVX512')
    os.environ.setdefault('OMP_NUM_THREADS', '1')  # joblib owns parallelism
    os.environ.setdefault('OPENBLAS_CORETYPE', 'SKYLAKEX')
    print('Local AVX-512 hints set (MKL_ENABLE_INSTRUCTIONS=AVX512, OPENBLAS_CORETYPE=SKYLAKEX)')

if not local:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
! pip install lightgbm scikit-learn scikit-learn-intelex

In [7]:
if local :
# 1. Apply the AMD acceleration patch FIRST
    # 1. Trigger the AVX-512 level optimizations
    from sklearnex import patch_sklearn
    patch_sklearn()



In [8]:
from pathlib import Path
if local:
    root = Path('data')
    if not root.exists():
        root = Path('.data')
else:
    root = Path('/content/drive/MyDrive/ml/CORRECTEDv3')

DATA_PATHS = {
    'Chongqing': root / 'delivery_features_chongqing.parquet',
    'Shanghai': root / 'delivery_features_shanghai.parquet',
    'Hangzhou': root / 'delivery_features_hangzhou.parquet',
}

OUTPUT_DIR = root / 'causal_graphs' / 'pc_delivery'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
STATIC_FEATURES = [

    # Structural
    "pickup_destination_distance",
    "batch_size",
    "batch_rank_dispatch",
    "same_aoi_share_in_batch",
    "isolated_delivery",
    "distance_to_batch_centroid",
    "typecode_cb",

    # Operational
    "courier_eta_ewm",
    "gps_points",
    "speed_mean_15m",
    "speed_std_15m",
    "distance_travelled_15m",
    "coverage_ratio",
    "gps_gap_min",
    "idle_fraction",
    "is_trajectory_available",

    # Environment
    "WSI",
    "temperature_2m",
    "precipitation",
    "windspeed_10m",
    "spatial_congestion_daily",
    "spatial_congestion_norm",

    # Time
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_holiday",
    "is_holiday_eve"
]




In [10]:
import pandas as pd

for city, path in DATA_PATHS.items():
    print(f"Processing {city} data...")
    try:
        df = pd.read_parquet(path)
        missing_features = [feature for feature in STATIC_FEATURES if feature not in df.columns]
        if not missing_features:
            print(f"  All STATIC_FEATURES are present in {city} dataframe.")
        else:
            print(f"  The following STATIC_FEATURES are missing in {city} dataframe: {missing_features}")
    except FileNotFoundError:
        print(f"  Error: File not found at {path}")
    except Exception as e:
        print(f"  An error occurred while processing {city} data: {e}")
print("\nFinished checking all dataframes.")

Processing Chongqing data...
  All STATIC_FEATURES are present in Chongqing dataframe.
Processing Shanghai data...
  All STATIC_FEATURES are present in Shanghai dataframe.
Processing Hangzhou data...
  All STATIC_FEATURES are present in Hangzhou dataframe.

Finished checking all dataframes.


In [16]:
DYNAMIC_FEATURES = [

    "dist_from_current",
    "remaining_orders",
    "batch_progress",
    "elapsed_route_time",
    "last_duration",
    "current_hour",
    "cumulative_distance"
]

In [33]:
import polars as pl

DYNAMIC_FEATURES = [
    "dist_from_current",
    "remaining_orders",
    "batch_progress",
    "elapsed_route_time",
    "last_duration",
    "current_hour",
    "cumulative_distance"
]

def compute_dynamic_features_polars(df: pl.DataFrame) -> pl.DataFrame:
    # Ensure datetime columns are in datetime format
    df = df.with_columns([
        pl.col('receipt_time').cast(pl.Datetime),
        pl.col('sign_time').cast(pl.Datetime)
    ])

    # Sort by batch_id and batch_rank_actual (0-indexed)
    sort_keys = ['batch_id', 'batch_rank_actual']
    df_sorted = df.sort(sort_keys)

    # 1. remaining_orders = (batch size - 1) - delivery index
    df_sorted = df_sorted.with_columns(
        (pl.col('batch_size').cast(pl.Int64) - 1 - pl.col('batch_rank_actual').cast(pl.Int64)).clip(lower_bound=0).alias('remaining_orders')
    )

    # 2. batch_progress = delivery index / (batch size - 1)
    df_sorted = df_sorted.with_columns(
        pl.when(pl.col('batch_size') <= 1)
        .then(pl.lit(0.0, dtype=pl.Float64))
        .otherwise(pl.col('batch_rank_actual').cast(pl.Float64) / (pl.col('batch_size').cast(pl.Float64) - 1))
        .alias('batch_progress')
    )

    # Helper calculations for incremental duration
    duration_from_receipt = (
        (pl.col('sign_time') - pl.col('receipt_time')).dt.total_minutes()
        .fill_null(0.0)
        .clip(lower_bound=0)
    )

    duration_from_prev_sign = (
        (pl.col('sign_time') - pl.col('sign_time').shift(1).over('batch_id')).dt.total_minutes()
        .fill_null(0.0)
        .clip(lower_bound=0)
    )

    # 3. Incremental Duration (0-indexed check)
    df_sorted = df_sorted.with_columns(
        pl.when(pl.col('batch_rank_actual') == 0)
        .then(duration_from_receipt)
        .otherwise(duration_from_prev_sign)
        .alias('incremental_duration')
    )

    # 4. elapsed_route_time
    df_sorted = df_sorted.with_columns(
        (pl.col('incremental_duration').cum_sum().over('batch_id') - pl.col('incremental_duration')).alias('elapsed_route_time')
    )

    # 5. Current Hour
    df_sorted = df_sorted.with_columns(
        (pl.col('receipt_time') + pl.duration(minutes=pl.col('elapsed_route_time'))).alias('current_time')
    )
    df_sorted = df_sorted.with_columns(
        (pl.col('current_time').dt.hour() + pl.col('current_time').dt.minute().cast(pl.Float64)/60).alias('current_hour')
    )

    # 6. last_duration
    df_sorted = df_sorted.with_columns(
        pl.col('incremental_duration')
        .shift(1)
        .fill_null(0.0)
        .over('batch_id')
        .alias('last_duration')
    )

    # 7. dist_from_current
    df_sorted = df_sorted.with_columns([
        pl.col('poi_lng').shift(1).over('batch_id').alias('prev_lng'),
        pl.col('poi_lat').shift(1).over('batch_id').alias('prev_lat')
    ])

    df_sorted = df_sorted.with_columns(
        (
            ((pl.col('poi_lng') - pl.col('prev_lng'))**2 + (pl.col('poi_lat') - pl.col('prev_lat'))**2).sqrt()
        )
        .fill_null(pl.col('pickup_destination_distance'))
        .alias('dist_from_current')
    )

    # 8. cumulative_distance
    df_sorted = df_sorted.with_columns(
        (pl.col('dist_from_current').cum_sum().over('batch_id') - pl.col('dist_from_current')).alias('cumulative_distance')
    )

    # Clean up temporary columns
    df_final = df_sorted.drop(['current_time', 'prev_lng', 'prev_lat', 'incremental_duration'])
    return df_final

processed_dfs = {}
for city, path in DATA_PATHS.items():
    print(f"\nProcessing {city} data...")
    try:
        df = pl.read_parquet(path)
        processed_df = compute_dynamic_features_polars(df)
        processed_dfs[city] = processed_df
        print(f"Finished {city}. New columns added.")
    except Exception as e:
        print(f"Error processing {city}: {e}")


Processing Chongqing data...
Finished Chongqing. New columns added.

Processing Shanghai data...
Finished Shanghai. New columns added.

Processing Hangzhou data...
Finished Hangzhou. New columns added.


In [35]:
# Verification check for 0-indexed rank logic NO NEED TO RUN FOR NORMAL FLOW
city_sample = 'Chongqing'
if city_sample in processed_dfs:
    df_check = processed_dfs[city_sample]
    # Look at a specific batch with more than 1 order to verify ranges
    multi_order_batches = df_check.filter(pl.col('batch_size') > 1).select('batch_id').unique().head(1)
    if not multi_order_batches.is_empty():
        bid = multi_order_batches.item()
        print(f"Verification for Batch ID: {bid}")
        print(df_check.filter(pl.col('batch_id') == bid).select([
            'batch_id', 'batch_size', 'batch_rank_actual', 'remaining_orders', 'batch_progress'
        ]))

Verification for Batch ID: 0008c2b6a2314db8715301b7eeeebc5a__37e976ad4abb10da92c97c6b34f7f54f__318__1616055600
shape: (8, 5)
┌─────────────────────────────┬────────────┬───────────────────┬──────────────────┬────────────────┐
│ batch_id                    ┆ batch_size ┆ batch_rank_actual ┆ remaining_orders ┆ batch_progress │
│ ---                         ┆ ---        ┆ ---               ┆ ---              ┆ ---            │
│ str                         ┆ u32        ┆ u32               ┆ i64              ┆ f64            │
╞═════════════════════════════╪════════════╪═══════════════════╪══════════════════╪════════════════╡
│ 0008c2b6a2314db8715301b7eee ┆ 8          ┆ 0                 ┆ 7                ┆ 0.0            │
│ ebc…                        ┆            ┆                   ┆                  ┆                │
│ 0008c2b6a2314db8715301b7eee ┆ 8          ┆ 1                 ┆ 6                ┆ 0.142857       │
│ ebc…                        ┆            ┆                   ┆   

In [28]:
import polars as pl
# Inspect columns to find the batch identifier
for city, path in DATA_PATHS.items():
    df_cols = pl.read_parquet(path).columns
    print(f"{city} columns: {df_cols}")
    break # Just check one to identify the column name

Chongqing columns: ['order_id', 'from_dipan_id', 'delivery_user_id', 'poi_lng', 'poi_lat', 'aoi_id', 'typecode', 'receipt_time', 'receipt_lng', 'receipt_lat', 'sign_time', 'ds', 'eta_mins', 'pickup_destination_distance', 'batch_size', 'isolated_delivery', 'same_aoi_share_in_batch', 'batch_rank_dispatch', 'batch_rank_actual', 'batch_id', 'distance_to_batch_centroid', 'gps_points', 'speed_mean_15m', 'speed_std_15m', 'distance_travelled_15m', 'idle_fraction', 'coverage_ratio', 'ds_right', 'last_gps_time', 'last_x', 'last_y', 'gps_gap_min', 'courier_eta_ewm', 'remaining_haul_distance', 'is_trajectory_available', 'hour', 'weekday', 'receipt_date', 'hour_sin', 'hour_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve', 'typecode_grouped_missing', 'typecode_grouped_other', 'typecode_grouped_type_1', 'typecode_grouped_type_2', 'typecode_cb', 'grid_x', 'grid_y', 'time_window', 'spatial_congestion_daily', 'spatial_congestion_norm', 'city_right', 'datetime', 'temperature_2m', 'precipitation', 'rain

## Sanity Check: Dynamic Features

In [29]:
for city, df_processed in processed_dfs.items():
    print(f"\n--- Sanity Check for {city} ---")
    for feature in DYNAMIC_FEATURES:
        if feature in df_processed.columns:
            print(f"Feature: {feature}")

            # Check for null values
            null_count = df_processed.select(pl.col(feature).is_null().sum()).item()
            if null_count > 0:
                print(f"  - Found {null_count} null values.")

            # Check for infinite values
            if df_processed[feature].dtype in [pl.Float32, pl.Float64]:
                infinite_count = df_processed.select(pl.col(feature).is_infinite().sum()).item()
                if infinite_count > 0:
                    print(f"  - Found {infinite_count} infinite values.")

            # Display basic statistics
            try:
                stats = df_processed.select(
                    pl.col(feature).mean().alias('mean'),
                    pl.col(feature).std().alias('std'),
                    pl.col(feature).min().alias('min'),
                    pl.col(feature).max().alias('max')
                ).row(0, named=True)
                print(f"  - Statistics: Mean={stats['mean']:.4f}, Std={stats['std']:.4f}, Min={stats['min']:.4f}, Max={stats['max']:.4f}")
            except Exception as e:
                print(f"  - Could not compute statistics: {e}")
        else:
            print(f"Feature: {feature} not found in {city} DataFrame.")


--- Sanity Check for Chongqing ---
Feature: dist_from_current
  - Statistics: Mean=1071.2385, Std=2609.6057, Min=0.0000, Max=47985.3203
Feature: remaining_orders
  - Statistics: Mean=4.8172, Std=4.0259, Min=1.0000, Max=33.0000
Feature: batch_progress
  - Statistics: Mean=0.1965, Std=0.4285, Min=-1.0000, Max=0.9394
Feature: elapsed_route_time
  - Statistics: Mean=115.9490, Std=142.6878, Min=0.0000, Max=1416.0000
Feature: last_duration
  - Statistics: Mean=31.0553, Std=61.7770, Min=0.0000, Max=1015.0000
Feature: current_hour
  - Statistics: Mean=12.9880, Std=3.4478, Min=0.2000, Max=23.2000
Feature: cumulative_distance
  - Statistics: Mean=4766.8784, Std=17426.0082, Min=0.0000, Max=435376.9660

--- Sanity Check for Shanghai ---
Feature: dist_from_current
  - Statistics: Mean=743.9231, Std=1263.3805, Min=0.0000, Max=18859.3745
Feature: remaining_orders
  - Statistics: Mean=4.2854, Std=3.2606, Min=1.0000, Max=25.0000
Feature: batch_progress
  - Statistics: Mean=0.1779, Std=0.4250, Min=-1.0